<a href="https://colab.research.google.com/github/SamsGibsP/WorkshopLPOSI_Sesi3/blob/main/Hands_On_Sesi_3_Workshop_Annotated_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔧 Persiapan: Import Library & Konfigurasi

Sebelum mulai analisis, kita perlu mengimpor library Python yang akan dipakai sepanjang notebook ini, yaitu:
- `numpy` & `pandas` → untuk manipulasi data (angka & tabel)
- `matplotlib` → untuk membuat visualisasi/grafik
- `statsmodels` → untuk membangun model regresi (OLS) beserta uji-uji statistiknya (VIF, Durbin-Watson, dll.)
- `scipy.stats` → untuk uji statistik tambahan (mis. Shapiro-Wilk)
- `sklearn` (scikit-learn) → untuk Ridge Regression, Lasso Regression, split data train-test, dan menghitung metrik evaluasi (RMSE, R²)

Cell berikutnya juga mengatur path/lokasi file dataset (`DATA_PATH`) dan folder tempat menyimpan output grafik (`OUT_DIR`). Ini murni konfigurasi, belum ada analisis di sini.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from scipy import stats as scipy_stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, linear_rainbow
from statsmodels.formula.api import ols

In [ ]:
from sklearn.linear_model import Lasso, LassoCV, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from google.colab import files

print("\U0001F4C2 Silakan klik tombol di bawah untuk upload file dataset "
      "(Dataset_Workshop_LPOSI2026_Sesi3.csv):")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

OUT_DIR = "/content/outputs"

📂 Silakan klik tombol di bawah untuk upload file dataset (Dataset_Workshop_LPOSI2026_Sesi3.csv):


1. Data Profiling

#### 📌 Apa yang dilakukan di sini?

Cell ini **memuat (load) dataset** dari file CSV ke dalam sebuah tabel (`DataFrame`) bernama `df`, lalu membuang kolom `Keterangan` karena isinya banyak kosong dan tidak dipakai dalam analisis.

Selanjutnya, ditampilkan ringkasan awal:
- Berapa jumlah observasi (baris/bulan) dan jumlah kolom (variabel)
- Variabel dependen (Y) yang ingin kita jelaskan/prediksi, yaitu `Omzet_Juta`
- Cuplikan 5 baris pertama data, sekadar untuk memastikan data terbaca dengan benar

Tahap ini disebut **Data Profiling** yaitu langkah awal wajib sebelum modeling, untuk memastikan kita paham bentuk dan struktur data yang akan diolah.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Keterangan"], errors="ignore")  # kolom catatan, banyak kosong

print(f"Jumlah observasi : {df.shape[0]} bulan")
print(f"Jumlah kolom      : {df.shape[1]}")
print("\nVariabel dependen (Y) : Omzet_Juta")
print("\nCuplikan data:")
print(df.head())

In [ ]:
# 1. Buat kolom datetime dari kolom 'Tahun' dan 'Bulan'
df['Tanggal'] = pd.to_datetime(
    df['Tahun'].astype(str) + '-' + df['Bulan'].astype(str) + '-01'
)

# 2. Urutkan berdasarkan waktu secara kronologis
df = df.sort_values(by='Tanggal').reset_index(drop=True)

# 3. Jadikan kolom 'Tanggal' sebagai Index dan tetapkan frekuensi bulanan ('MS')
df_ts = df.set_index('Tanggal')
df_ts.index.freq = 'MS'  # MS = Month Start (Awal Bulan)

# 4. Tampilkan informasi DataFrame Time Series
print("=== STRUKTUR TIME SERIES DATAFRAME ===")
print(df_ts.info())
print("\nCuplikan Data dengan DatetimeIndex:")
display(
    df_ts[
        ['t_Waktu', 'D_Ramadhan', 'D_AkhirTahun', 'Omzet_Juta']
    ].head()
)

#### 📌 Apa yang dilakukan di sini?

Cell ini membuat **grafik deret waktu (time series plot)** dari `Omzet_Juta` terhadap `t_Waktu` (urutan bulan ke-1 sampai ke-120).

Tujuannya untuk melihat secara visual:
- Apakah omzet punya **tren** (kecenderungan naik/turun dalam jangka panjang)?
- Apakah ada pola **musiman (seasonality)** yang berulang setiap periode tertentu (mis. tiap tahun)?

Grafik ini disimpan sebagai file gambar (`.png`) di folder output, lalu ditampilkan langsung di notebook.

In [ ]:
import os

# Create the output directory if it doesn't exist
os.makedirs(OUT_DIR, exist_ok=True)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["Tanggal"], df["Omzet_Juta"], color="firebrick", linewidth=1.3)
ax.set_title("Deret Waktu Omzet Bulanan (menunjukkan Trend & Seasonality)")
ax.set_xlabel("Periode Waktu (t)")
ax.set_ylabel("Omzet (Juta Rp)")
ax.grid(alpha=0.3)
fig.tight_layout()

# Simpan ke file terlebih dahulu
fig.savefig(f"{OUT_DIR}/01_time_series_omzet.png", dpi=150)

# Tampilkan grafik ke layar/notebook
plt.show()

2. Simple Linear Regression

### 📈 2. Simple Linear Regression

#### 📌 Apa yang dilakukan di sini?

Ini adalah model regresi paling sederhana: hanya menggunakan **satu variabel prediktor (X)**, yaitu `t_Waktu`, untuk menjelaskan `Omzet_Juta` (Y).

- `sm.add_constant(...)` menambahkan kolom "intercept" (konstanta) ke X — ini wajib di `statsmodels` agar model punya nilai intercept (a), bukan hanya slope (b).
- `sm.OLS(y, X_simple).fit()` membangun model regresi linear menggunakan metode **Ordinary Least Squares (OLS)**, yaitu metode standar untuk mencari garis regresi terbaik (meminimalkan jumlah kuadrat error).

Model ini nantinya menjawab pertanyaan: *"Apakah omzet cenderung naik/turun seiring berjalannya waktu, dan seberapa besar?"*

In [ ]:
X_simple = sm.add_constant(df["t_Waktu"])
y = df["Omzet_Juta"]
model_simple = sm.OLS(y, X_simple).fit()

#### 📌 Apa yang dilakukan di sini?

Cell ini **mengambil angka-angka penting** dari hasil model (`model_simple`) yang sudah dibangun sebelumnya, yaitu:
- `a` → nilai **intercept** (perkiraan omzet saat t_Waktu = 0)
- `b` → nilai **slope/koefisien** (rata-rata perubahan omzet setiap 1 periode waktu berlalu)
- `r2_simple` → nilai **R-squared** (seberapa besar persentase variasi omzet yang bisa dijelaskan oleh waktu saja)
- `pval_b` → **p-value** dari slope (dipakai untuk menguji apakah pengaruh waktu terhadap omzet signifikan secara statistik atau tidak)

In [ ]:
a = model_simple.params["const"]
b = model_simple.params["t_Waktu"]
r2_simple = model_simple.rsquared
pval_b = model_simple.pvalues["t_Waktu"]

#### 📌 Apa yang dilakukan di sini?

Cell ini menampilkan **ringkasan lengkap model** (`model_simple.summary()`) — output standar dari `statsmodels` yang berisi koefisien, R², p-value, F-statistic, dll.

Di bawahnya, dicetak interpretasi dalam bahasa yang lebih mudah dipahami:
- Berapa rata-rata perubahan omzet setiap 1 periode waktu
- Berapa persen variasi omzet yang dijelaskan oleh waktu (R²)
- Apakah pengaruh waktu terhadap omzet **signifikan** (p-value < 0.05) atau tidak

💡 **Catatan konsep**: p-value < 0.05 secara umum diartikan sebagai "pengaruh variabel tersebut terhadap Y cukup meyakinkan secara statistik, bukan kebetulan".

In [ ]:
print(model_simple.summary())
print(f"\nInterpretasi: Setiap 1 periode waktu berlalu, Omzet berubah rata-rata "
      f"{b:.2f} juta Rupiah, dengan nilai awal (intercept) sebesar {a:.2f}.")
print(f"R-squared         : {r2_simple:.4f}  -> {r2_simple*100:.1f}% variasi Omzet dijelaskan oleh waktu")
print(f"P-value (slope)   : {pval_b:.4g}  -> "
      f"{'Signifikan (p<0.05)' if pval_b < 0.05 else 'Tidak signifikan (p>=0.05)'}")

#### 📌 Apa yang dilakukan di sini?

Cell ini membuat **scatter plot** (titik-titik data aktual) omzet terhadap waktu, lalu menimpanya dengan **garis regresi** hasil model (`model_simple.predict(...)`).

Grafik ini membantu kita **melihat secara visual** seberapa baik garis regresi sederhana ini "mengikuti" pola data aktual — sebagai pembanding sebelum nanti masuk ke model yang lebih kompleks (multiple regression).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["t_Waktu"], df["Omzet_Juta"], alpha=0.6, label="Data aktual")
ax.plot(
    df["t_Waktu"],
    model_simple.predict(X_simple),
    color="red",
    linewidth=2,
    label=f"Omzet = {a:.2f} + {b:.2f}(t)",
)
ax.set_xlabel("Waktu (t)")
ax.set_ylabel("Omzet (Juta Rp)")
ax.set_title("Simple Linear Regression: Omzet vs Waktu")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

# 1. Simpan gambar ke file
fig.savefig(f"{OUT_DIR}/02_simple_regression.png", dpi=150)

# 2. Tampilkan langsung di layar Colab (tanpa plt.close sebelumnya)
plt.show()

3. Memodelkan Trends dan Seasonal Effect

### 📅 3. Memodelkan Trend dan Seasonal Effect

#### 📌 Apa yang dilakukan di sini?

Model sebelumnya (simple regression) hanya menangkap **tren** (lewat `t_Waktu`). Di sini, model diperluas dengan menambahkan dua **variabel dummy** yang menangkap efek musiman:
- `D_Ramadhan` (1 jika bulan tersebut Ramadhan, 0 jika bukan)
- `D_AkhirTahun` (1 jika bulan tersebut termasuk periode akhir tahun, 0 jika bukan)

Jadi model ini sekarang menjelaskan omzet berdasarkan **tren jangka panjang + efek musiman**, bukan hanya waktu saja.

In [ ]:
seasonal_features = ["t_Waktu", "D_Ramadhan", "D_AkhirTahun"]
X_season = sm.add_constant(df[seasonal_features])
model_season = sm.OLS(y, X_season).fit()

#### 📌 Apa yang dilakukan di sini?

Cell ini menampilkan ringkasan model beserta interpretasi khusus untuk **koefisien variabel dummy**:
- Koefisien `D_Ramadhan` → rata-rata **tambahan** omzet pada bulan Ramadhan dibanding bulan biasa (dengan asumsi variabel lain tetap)
- Koefisien `D_AkhirTahun` → rata-rata tambahan omzet saat periode akhir tahun
- Koefisien `t_Waktu` → komponen tren jangka panjang (setelah efek musiman dikeluarkan)

Nilai p-value di sebelah masing-masing koefisien menunjukkan apakah efek musiman tersebut signifikan secara statistik.

In [ ]:
print(model_season.summary())
print("\nInterpretasi koefisien dummy:")
print(f"  D_Ramadhan   : {model_season.params['D_Ramadhan']:.2f} "
      f"(p={model_season.pvalues['D_Ramadhan']:.4g}) -> rata-rata tambahan Omzet saat bulan Ramadhan")
print(f"  D_AkhirTahun : {model_season.params['D_AkhirTahun']:.2f} "
      f"(p={model_season.pvalues['D_AkhirTahun']:.4g}) -> rata-rata tambahan Omzet saat akhir tahun")
print(f"  t_Waktu      : {model_season.params['t_Waktu']:.2f} -> komponen trend jangka panjang")
print(f"R-squared model trend+seasonal : {model_season.rsquared:.4f}")

4. Multiple Linear Regression

### 🧮 4. Multiple Linear Regression

#### 📌 Apa yang dilakukan di sini?

Di sinilah kita mulai masuk ke **inti Multiple Linear Regression**: menggunakan **banyak variabel prediktor sekaligus**, bukan cuma satu atau dua seperti sebelumnya.

Cell ini menyiapkan daftar `candidate_features`, yaitu semua kolom di dataset **kecuali**:
- `ID`, `Tahun`, `Bulan`, `NamaBulan` → bukan variabel prediktor, hanya identitas/waktu
- `Omzet_Juta` → ini variabel Y (target), bukan prediktor

Sisanya (harga jual, biaya produksi, inflasi, dummy event, dll.) akan jadi kandidat variabel X yang dimasukkan ke model.

In [ ]:
# Tambahkan "Tanggal" ke dalam exclude_cols bersama kolom non-prediktor lainnya
exclude_cols = ["ID", "Tahun", "Bulan", "NamaBulan", "Tanggal", "Omzet_Juta"]

# Buat list candidate_features
candidate_features = [c for c in df.columns if c not in exclude_cols]

print(f"Jumlah kandidat variabel prediktor (k) : {len(candidate_features)}")
print("Daftar kandidat:", candidate_features)

#### 📌 Apa yang dilakukan di sini?

Cell ini membangun model regresi dengan **seluruh kandidat variabel** (`candidate_features`) dimasukkan sekaligus ke dalam model (`model_full`). Ini disebut **model "full"** karena belum ada proses seleksi variabel — semua variabel yang ada langsung dipakai.

In [ ]:
print(model_full.summary())

In [ ]:
numeric_candidate_features = [col for col in candidate_features if col != 'Tanggal']
X_full = sm.add_constant(df[candidate_features])
model_full = sm.OLS(y, X_full).fit()

#### 📌 Apa yang dilakukan di sini?

Cell ini menampilkan performa model full:
- **R-squared** → persentase variasi omzet yang dijelaskan oleh seluruh variabel
- **Adjusted R-squared** → versi R² yang sudah "dihukum" karena jumlah variabel yang banyak (lebih adil untuk membandingkan model dengan jumlah variabel berbeda)
- **F-statistic p-value** → menguji apakah model secara keseluruhan signifikan (minimal ada satu variabel yang berpengaruh)

⚠️ Ada catatan penting: model dengan **banyak variabel sekaligus** rawan mengalami **multikolinearitas** (variabel-variabel prediktor saling berkorelasi kuat satu sama lain), yang bisa membuat interpretasi koefisien jadi tidak stabil/menyesatkan. Ini akan dicek lebih lanjut di bagian **Robustness Test**.

In [ ]:
print(f"\nR-squared          : {model_full.rsquared:.4f}")
print(f"Adjusted R-squared : {model_full.rsquared_adj:.4f}")
print(f"F-statistic p-value: {model_full.f_pvalue:.4g}")
print("\n(Model penuh ini rawan MULTIKOLINEARITAS karena banyak prediktor saling "
      "berkorelasi -- lihat Bagian 6. Robustness Test)")

4. Metode Stepwise

### 🔍 Metode Stepwise

#### 📌 Apa yang dilakukan di sini?

Karena model full (semua variabel) berisiko multikolinearitas dan sulit diinterpretasi, kita perlu **menyeleksi variabel mana saja yang benar-benar relevan/signifikan**. Salah satu caranya adalah **Stepwise Regression**.

Cell ini **mendefinisikan fungsi** `stepwise_selection`, yang bekerja dengan menggabungkan dua langkah secara berulang:
- **Forward step**: coba tambahkan satu-per-satu variabel yang punya p-value paling kecil (paling signifikan) di bawah `threshold_in` (0.05)
- **Backward step (check)**: setelah suatu variabel masuk, cek ulang — kalau ada variabel di dalam model yang p-value-nya sudah tidak signifikan lagi (di atas `threshold_out` = 0.10), variabel itu dikeluarkan

Proses ini berulang (forward → backward → forward → ...) sampai tidak ada lagi perubahan (tidak ada variabel yang ditambah/dikeluarkan). Fungsi ini baru **didefinisikan** di sini, belum dijalankan.

In [ ]:
def stepwise_selection(X, y, threshold_in=0.05, threshold_out=0.10, verbose=True):
    """Implementasi Stepwise Regression (forward + backward check) berbasis p-value."""
    included = []
    step = 0
    while True:
        changed = False

        # ---- Forward step ----
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        for col in excluded:
            trial_cols = included + [col]
            model = sm.OLS(y, sm.add_constant(X[trial_cols])).fit()
            new_pval[col] = model.pvalues[col]

        if not new_pval.empty and new_pval.min() < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            step += 1
            if verbose:
                print(f"  Step {step} [FORWARD]  + tambah '{best_feature}' "
                      f"(p-value={new_pval.min():.4g})")

        # ---- Backward step (backward check) ----
        if included:
            model = sm.OLS(y, sm.add_constant(X[included])).fit()
            pvalues = model.pvalues.iloc[1:]  # exclude const
            worst_pval = pvalues.max()
            if worst_pval > threshold_out:
                worst_feature = pvalues.idxmax()
                included.remove(worst_feature)
                changed = True
                step += 1
                if verbose:
                    print(f"  Step {step} [BACKWARD] - keluarkan '{worst_feature}' "
                          f"(p-value={worst_pval:.4g})")

        if not changed:
            break

    return included

#### 📌 Apa yang dilakukan di sini?

Cell ini **menjalankan** fungsi `stepwise_selection` yang tadi didefinisikan, menggunakan seluruh `candidate_features` sebagai kandidat awal. Prosesnya akan mencetak log setiap langkah (variabel apa yang ditambah/dikeluarkan) sampai konvergen (stabil).

In [ ]:
selected_features = stepwise_selection(
    df[candidate_features], y, threshold_in=0.05, threshold_out=0.1
)

#### 📌 Apa yang dilakukan di sini?

Cell ini menampilkan **daftar final variabel** yang terpilih hasil proses stepwise (`selected_features`) — ini adalah variabel-variabel yang dianggap paling relevan/signifikan untuk menjelaskan omzet, hasil seleksi otomatis tadi.

In [ ]:
print(f"\nVariabel terpilih hasil Stepwise ({len(selected_features)} variabel):")
for f in selected_features:
    print(f"  - {f}")

#### 📌 Apa yang dilakukan di sini?

Cell ini membangun **model regresi baru** (`model_step`), tapi kali ini hanya menggunakan variabel-variabel hasil seleksi stepwise (`selected_features`), bukan semua kandidat. Ringkasan model ini bisa dibandingkan dengan model full sebelumnya — biasanya jumlah variabelnya lebih sedikit, tapi lebih stabil dan mudah diinterpretasi.

In [ ]:
X_step = sm.add_constant(df[selected_features])
model_step = sm.OLS(y, X_step).fit()
print()
print(model_step.summary())

5. Robustness Test

### ✅ 5. Robustness Test

Setelah punya model "final" hasil stepwise, kita perlu **menguji apakah model ini valid/layak dipercaya**, dengan memeriksa beberapa asumsi dasar regresi linear.

#### 📌 Apa yang dilakukan di sini? — Uji Multikolinearitas (VIF)

Cell ini menghitung **VIF (Variance Inflation Factor)** untuk setiap variabel di `selected_features`. VIF mengukur seberapa besar suatu variabel prediktor "dijelaskan" oleh variabel prediktor lainnya.

- VIF tinggi → variabel tersebut sangat berkorelasi dengan variabel lain di model (multikolinearitas), yang bisa membuat koefisien regresi jadi tidak stabil/sulit diinterpretasi secara individual.

In [ ]:
X_vif = sm.add_constant(df[selected_features])

In [ ]:
vif_data = pd.DataFrame()
vif_data["Variabel"] = selected_features
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(1, X_vif.shape[1])  # Mulai dari indeks 1 untuk melewati 'const'
]

#### 📌 Apa yang dilakukan di sini?

Cell ini mendefinisikan fungsi bantu `vif_interpretasi` untuk menerjemahkan angka VIF menjadi kategori yang mudah dibaca (Tidak ada korelasi / Moderat / Cukup tinggi / Tinggi). Ini semacam "kamus" ambang batas VIF yang umum dipakai.

In [ ]:
def vif_interpretasi(v):
    if v < 1.01:
        return "Tidak ada multikolinearitas"
    elif v < 5:
        return "Korelasi rendah-moderat (Aman)"
    elif v < 10:
        return "Multikolinearitas moderat-tinggi (Perlu diwaspadai)"
    else:
        return "Multikolinearitas TINGGI (Perlu tindakan/regularisasi)"

#### 📌 Apa yang dilakukan di sini?

Cell ini menampilkan **tabel hasil VIF** lengkap dengan interpretasinya untuk setiap variabel di `selected_features`, supaya kita bisa langsung lihat variabel mana (jika ada) yang punya masalah multikolinearitas tinggi.

In [ ]:
vif_data["Interpretasi"] = vif_data["VIF"].apply(vif_interpretasi)
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
print("=== TABEL HASIL CENTERED VIF (ROBUSTNESS TEST) ===")
print(vif_data.to_string(index=False))

#### 📌 Apa yang dilakukan di sini? — Uji Normalitas Residual (Shapiro-Wilk)

Salah satu asumsi regresi linear adalah **residual (error) harus berdistribusi normal**. Cell ini menjalankan **uji Shapiro-Wilk** terhadap residual model (`model_step.resid`).

- p-value ≥ 0.05 → residual dianggap berdistribusi normal (asumsi terpenuhi)
- p-value < 0.05 → residual **tidak** normal (asumsi ini mungkin dilanggar)

In [ ]:
# Uji Shapiro-Wilk
resid = model_step.resid
sw_stat, sw_p = scipy_stats.shapiro(resid)
print(f"Shapiro-Wilk statistic = {sw_stat:.4f}, p-value = {sw_p:.4g}")
print("-> " + ("Residual berdistribusi NORMAL (p>=0.05)" if sw_p >= 0.05
                else "Residual TIDAK berdistribusi normal (p<0.05)"))

#### 📌 Apa yang dilakukan di sini? — Uji Homoskedastisitas (Breusch-Pagan)

Asumsi lain regresi linear: **varians residual harus konstan** di semua level prediksi (disebut homoskedastisitas). Kebalikannya, **heteroskedastisitas**, terjadi kalau varians error membesar/mengecil secara sistematis.

Cell ini menjalankan **uji Breusch-Pagan** untuk mendeteksi ini:
- p-value ≥ 0.05 → homoskedastis (asumsi terpenuhi, aman)
- p-value < 0.05 → terindikasi heteroskedastisitas (perlu perhatian, misalnya pertimbangkan transformasi variabel)

In [ ]:
# Uji Homoskedastisitas
bp_test = sm.stats.diagnostic.het_breuschpagan(resid, X_step)
bp_labels = ["LM Statistic", "LM p-value", "F-Statistic", "F p-value"]
bp_result = dict(zip(bp_labels, bp_test))
for k, v in bp_result.items():
    print(f"  {k:15s}: {v:.4g}")
print("-> " + ("HOMOSKEDASTIS, tidak ada pola varian error (p>=0.05)"
                if bp_result["LM p-value"] >= 0.05
                else "Terindikasi HETEROSKEDASTISITAS (p<0.05)"))

#### 📌 Apa yang dilakukan di sini? — Uji Autokorelasi (Durbin-Watson)

Karena data ini berbentuk **deret waktu** (time series bulanan), penting untuk mengecek apakah residualnya **saling berkorelasi antar waktu** (autokorelasi) — misalnya, error bulan ini mirip dengan error bulan sebelumnya.

Cell ini menghitung **statistik Durbin-Watson**:
- Nilai mendekati **2** → tidak ada indikasi autokorelasi (bagus)
- Nilai **< 1.5** → indikasi autokorelasi positif
- Nilai **> 2.5** → indikasi autokorelasi negatif

In [ ]:
# Uji Autokorelasi
dw_stat = durbin_watson(resid)
print(f"Durbin-Watson statistic = {dw_stat:.4f}")
if 1.5 <= dw_stat <= 2.5:
    dw_ket = "Tidak ada indikasi autokorelasi (mendekati 2)"
elif dw_stat < 1.5:
    dw_ket = "Indikasi autokorelasi POSITIF"
else:
    dw_ket = "Indikasi autokorelasi NEGATIF"
print(f"-> {dw_ket}")

#### 📌 Apa yang dilakukan di sini? — Deteksi Outlier

Cell ini menghitung **standardized residual** untuk tiap observasi (seberapa jauh nilai aktual menyimpang dari nilai prediksi, dalam satuan standar deviasi).

Observasi dengan `|Standardized Residual| > 3` dianggap **outlier ekstrem** — kemungkinan bulan-bulan dengan kejadian tidak biasa (mis. lonjakan omzet karena promo viral, atau penurunan drastis karena gangguan operasional) yang tidak bisa dijelaskan dengan baik oleh model.

In [ ]:
# Deteksi Outlier
influence = model_step.get_influence()
std_resid = influence.resid_studentized_internal
outlier_df = pd.DataFrame({
    "Obs": df.index,
    "t_Waktu": df["t_Waktu"],
    "Omzet_Aktual": y.values,
    "Omzet_Fit": model_step.fittedvalues.values,
    "Std_Resid": std_resid
})
outliers = outlier_df[outlier_df["Std_Resid"].abs() > 3]
print(f"Jumlah observasi dengan |Standardized Residual| > 3 : {len(outliers)}")
if len(outliers) > 0:
    print(outliers.to_string(index=False))
else:
    print("Tidak ditemukan outlier ekstrem pada model hasil stepwise.")

In [ ]:
# =======================================================================
# 1. PENDEKATAN A: TRANSFORMASI LOGARITMA LOG(Y)
# =======================================================================
y_log = np.log(df["Omzet_Juta"])
X_step = sm.add_constant(df[selected_features])

# Fit model dengan target log(Y)
model_log = sm.OLS(y_log, X_step).fit()
resid_log = model_log.resid

# Uji Normalitas Residual Model Log
sw_stat_log, sw_p_log = scipy_stats.shapiro(resid_log)

print(f"\n[Pendekatan A] Model Transformasi Log(Y):")
print(f"  - Shapiro-Wilk Stat : {sw_stat_log:.4f}")
print(f"  - P-value           : {sw_p_log:.4g}")
print(
    f"  - Kesimpulan        : {'Residual NORMAL (p>=0.05)' if sw_p_log >= 0.05 else 'Residual Belum Normal (p<0.05)'}"
)
print(f"  - R-squared (Log)   : {model_log.rsquared:.4f}")

In [ ]:
# =======================================================================
# 2. PENDEKATAN B: ANALISIS SENSITIVITAS (DENGAN vs TANPA OUTLIER EKSTREM)
# =======================================================================
# Identifikasi indeks outlier dari standardized residual > 3
influence = model_step.get_influence()
std_resid = influence.resid_studentized_internal
outlier_indices = df.index[np.abs(std_resid) > 3].tolist()

print(f"\n[Pendekatan B] Analisis Sensitivitas Outlier:")
print(f"  - Indeks Outlier Terdeteksi: {outlier_indices}")

# Buat subset data tanpa outlier ekstrem
df_clean = df.drop(index=outlier_indices).reset_index(drop=True)
X_step_clean = sm.add_constant(df_clean[selected_features])
y_clean = df_clean["Omzet_Juta"]

# Fit model OLS tanpa outlier
model_clean = sm.OLS(y_clean, X_step_clean).fit()
resid_clean = model_clean.resid

# Uji Normalitas Residual setelah outlier dibersihkan
sw_stat_clean, sw_p_clean = scipy_stats.shapiro(resid_clean)

print(f"  - Shapiro-Wilk Stat (Tanpa Outlier) : {sw_stat_clean:.4f}")
print(f"  - P-value (Tanpa Outlier)           : {sw_p_clean:.4g}")
print(
    f"  - Kesimpulan                        : {'Residual NORMAL (p>=0.05)' if sw_p_clean >= 0.05 else 'Residual Belum Normal'}"
)
print(f"  - R-squared (Tanpa Outlier)         : {model_clean.rsquared:.4f}")

# Tabel Perbandingan Koefisien
coeff_compare = pd.DataFrame({
    "Variabel": ["const"] + selected_features,
    "Koefisien (Full Data)": model_step.params.values,
    "Koefisien (Tanpa Outlier)": model_clean.params.values,
    "P-value (Tanpa Outlier)": model_clean.pvalues.values,
})
print("\n--- Perbandingan Stabilitas Koefisien Regresi ---")
print(coeff_compare.to_string(index=False))

In [ ]:
# Tambahkan variabel dummy intervensi ke dataset
df["D_Event30"] = (df["t_Waktu"] == 30).astype(int)
df["D_Event114"] = (df["t_Waktu"] == 114).astype(int)

# Fit ulang model dengan tambahan variabel intervensi
features_with_events = selected_features + ["D_Event30", "D_Event114"]
model_intervention = sm.OLS(y, sm.add_constant(df[features_with_events])).fit()

In [ ]:
print(model_intervention.summary())

In [ ]:
# =======================================================================
# 3. VISUALISASI DIAGNOSTIK NORMALITAS (Q-Q PLOT & HISTOGRAM)
# =======================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Q-Q Plot Model Awal (Stepwise Asli)
scipy_stats.probplot(model_step.resid, dist="norm", plot=axes[0, 0])
axes[0, 0].set_title(
    f"Q-Q Plot: Model Asli\n(Shapiro p={scipy_stats.shapiro(model_step.resid)[1]:.2e})",
    fontsize=11,
)
axes[0, 0].grid(alpha=0.3)

# 2. Histogram Residual Model Awal
axes[0, 1].hist(
    model_step.resid, bins=20, color="firebrick", edgecolor="black", alpha=0.7
)
axes[0, 1].axvline(x=0, color="black", linestyle="--")
axes[0, 1].set_title("Distribusi Residual: Model Asli", fontsize=11)
axes[0, 1].set_xlabel("Residual")
axes[0, 1].grid(alpha=0.3)

# 3. Q-Q Plot Model Tanpa Outlier
scipy_stats.probplot(resid_clean, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title(
    f"Q-Q Plot: Model Tanpa Outlier\n(Shapiro p={sw_p_clean:.4f})", fontsize=11
)
axes[1, 0].grid(alpha=0.3)

# 4. Histogram Residual Model Tanpa Outlier
axes[1, 1].hist(
    resid_clean, bins=20, color="forestgreen", edgecolor="black", alpha=0.7
)
axes[1, 1].axvline(x=0, color="black", linestyle="--")
axes[1, 1].set_title(
    "Distribusi Residual: Model Tanpa Outlier (Ternormalisasi)", fontsize=11
)
axes[1, 1].set_xlabel("Residual")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
resid_clean = model_intervention.resid
fitted_clean = model_intervention.fittedvalues
X_interv = sm.add_constant(df[features_with_events])

# Uji statistik formal pada model intervensi
sw_stat_clean, sw_p_clean = scipy_stats.shapiro(resid_clean)
bp_test_clean = het_breuschpagan(resid_clean, X_interv)
rb_stat_clean, rb_p_clean = linear_rainbow(model_intervention)
dw_stat_clean = durbin_watson(resid_clean)

NameError: name 'model_intervention' is not defined

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Normal Q-Q Plot (Normalitas Sempurna)
scipy_stats.probplot(resid_final, dist="norm", plot=axes[0, 0])
axes[0, 0].set_title(
    f"Normal Q-Q Plot (Model Intervensi)\nShapiro p-value = {sw_p_fin:.4f} (NORMAL)",
    fontsize=11,
)
axes[0, 0].grid(alpha=0.3)

# 2. Residuals vs Fitted Plot (Homoskedastisitas)
axes[0, 1].scatter(
    fitted_final, resid_final, alpha=0.7, color="navy", edgecolors="black"
)
axes[0, 1].axhline(y=0, color="red", linestyle="--", linewidth=1.5)
axes[0, 1].set_title(
    f"Residuals vs Fitted (Homoskedastisitas)\nBreusch-Pagan p-value = {bp_test_fin[1]:.4f}",
    fontsize=11,
)
axes[0, 1].set_xlabel("Fitted Values (Prediksi Omzet)")
axes[0, 1].set_ylabel("Residuals")
axes[0, 1].grid(alpha=0.3)

# 3. Scatter Plot Linearitas (Aktual vs Prediksi)
axes[1, 0].scatter(
    fitted_final, y, alpha=0.7, color="darkgreen", edgecolors="black"
)
min_val = min(fitted_final.min(), y.min())
max_val = max(fitted_final.max(), y.max())
axes[1, 0].plot(
    [min_val, max_val],
    [min_val, max_val],
    color="red",
    linestyle="--",
    label="Ideal 45° Line",
)
axes[1, 0].set_title(
    f"Scatter Linearitas (Aktual vs Prediksi)\nRainbow p-value = {rb_p_fin:.4f}",
    fontsize=11,
)
axes[1, 0].set_xlabel("Fitted Values (Prediksi)")
axes[1, 0].set_ylabel("Actual Values (Aktual)")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Residuals vs Waktu (Autokorelasi)
axes[1, 1].plot(
    df["t_Waktu"],
    resid_final,
    marker="o",
    color="purple",
    linewidth=1.2,
    alpha=0.7,
)
axes[1, 1].axhline(y=0, color="red", linestyle="--", linewidth=1.5)
axes[1, 1].set_title(
    f"Residuals vs Waktu (Bebas Autokorelasi)\nDurbin-Watson = {dw_stat_fin:.4f}",
    fontsize=11,
)
axes[1, 1].set_xlabel("Periode Waktu (t)")
axes[1, 1].set_ylabel("Residuals")
axes[1, 1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

Ridge Regression

### 🎯 Ridge & Lasso Regression (Regularisasi)

Bagian ini membandingkan model regresi biasa (OLS) dengan dua teknik **regularisasi**: **Ridge** dan **Lasso**. Regularisasi berguna terutama saat multikolinearitas tinggi, karena bisa membuat model lebih stabil dengan cara "menahan" koefisien agar tidak terlalu besar.

#### 📌 Apa yang dilakukan di sini?

Cell ini membagi data menjadi **data latih (train)** dan **data uji (test)** dengan proporsi 80:20 (`test_size=0.2`).

Tujuannya: model dilatih (fit) hanya menggunakan data train, lalu diuji performanya pada data test yang **belum pernah dilihat model** — ini cara yang lebih jujur untuk menilai seberapa baik model bisa memprediksi data baru, dibanding sekadar mengevaluasi pada data yang sama dipakai untuk training.

In [ ]:
# 1. Pastikan urut waktu dan ambil X dan y
df_sorted = df.sort_values(by="t_Waktu").reset_index(drop=True)
X_all = df_sorted[numeric_candidate_features] # Use numeric_candidate_features here
y = df_sorted["Omzet_Juta"]

# 2. Split kronologis (80% train = 96 bulan awal, 20% test = 24 bulan akhir)
split_idx = int(len(df_sorted) * 0.8)

X_train, X_test = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
t_test = df_sorted["t_Waktu"].iloc[split_idx:]

print(
    f"Data Train: t = 1 s/d {split_idx} ({len(X_train)} bulan) [MASA LALU]"
)
print(
    f"Data Test : t = {split_idx + 1} s/d {len(df_sorted)} ({len(X_test)} bulan) [MASA DEPAN]"
)

# 3. Standardisasi (Fit HANYA di train, lalu transform ke test)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [ ]:
X_all = df[candidate_features]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

#### 📌 Apa yang dilakukan di sini?

Cell ini melakukan **standardisasi (scaling)** pada variabel prediktor menggunakan `StandardScaler`, sehingga semua variabel punya skala yang sebanding (rata-rata 0, standar deviasi 1).

Ini **wajib** dilakukan sebelum Ridge/Lasso, karena kedua metode ini sensitif terhadap skala variabel — variabel dengan skala besar (mis. harga dalam ribuan rupiah) bisa "mendominasi" secara tidak adil dibanding variabel berskala kecil (mis. rating 1-5) kalau tidak distandarisasi dulu.

In [ ]:
scaler = StandardScaler()
# Ensure only numeric features are passed to the scaler
X_train_numeric = X_train[numeric_candidate_features]
X_test_numeric = X_test[numeric_candidate_features]
X_train_sc = scaler.fit_transform(X_train_numeric)
X_test_sc = scaler.transform(X_test_numeric)

#### 📌 Apa yang dilakukan di sini?

Cell ini melatih dua model regularisasi menggunakan data train yang sudah distandarisasi:
- **Ridge Regression** → menekan (mengecilkan) koefisien variabel yang kurang penting, tapi tidak pernah membuatnya persis nol
- **Lasso Regression** → bisa membuat koefisien variabel yang kurang penting menjadi **persis nol**, sehingga otomatis "menyeleksi" variabel juga (mirip stepwise, tapi caranya berbeda)

Parameter `alpha=1.0` mengatur seberapa kuat efek regularisasi ini diterapkan.

In [ ]:
# 1. Hyperparameter Tuning Ridge & Lasso dengan TimeSeriesSplit
cv_ts = TimeSeriesSplit(n_splits=5)
alphas = np.logspace(-3, 3, 50)
alphas_grid = np.logspace(-4, 4, 100)

In [ ]:
ridge = RidgeCV(
    alphas=alphas_grid,
    cv=cv_ts,
    scoring="neg_mean_squared_error"
).fit(X_train_sc, y_train)
lasso = LassoCV(
    alphas=alphas_grid,
    cv=cv_ts,
    max_iter=20000,
    random_state=42
).fit(X_train_sc, y_train)

print(f"Alpha Optimal Terpilih untuk Ridge : {ridge.alpha_:.5f}")
print(f"Alpha Optimal Terpilih untuk Lasso : {lasso.alpha_:.5f}")

#### 📌 Apa yang dilakukan di sini?

Sebagai pembanding, cell ini membangun ulang model **OLS biasa** (tanpa regularisasi) menggunakan data train yang **belum** distandarisasi (`X_train` asli), supaya nanti bisa dibandingkan performanya secara adil dengan Ridge dan Lasso di data test.

In [ ]:
ols_compare = sm.OLS(y_train, sm.add_constant(X_train[numeric_candidate_features])).fit()
X_test_ols = sm.add_constant(X_test[numeric_candidate_features], has_constant="add")

#### 📌 Apa yang dilakukan di sini?

Cell ini menghasilkan **prediksi** dari ketiga model (OLS full, Ridge, Lasso) terhadap data test (`X_test`) yang belum pernah dilihat masing-masing model saat training. Prediksi ini nanti dipakai untuk menghitung metrik evaluasi (RMSE & R²).

In [ ]:
pred_ols = ols_compare.predict(X_test_ols)
pred_ridge = ridge.predict(X_test_sc)
pred_lasso = lasso.predict(X_test_sc)

#### 📌 Apa yang dilakukan di sini?

Sebagai pembanding tambahan, cell ini juga membangun model OLS **khusus dengan variabel hasil stepwise saja** (`selected_features`), dilatih di data train dan diuji di data test — jadi kita punya total 4 model untuk dibandingkan: OLS Full, OLS Stepwise, Ridge, dan Lasso.

In [ ]:
X_train_step = sm.add_constant(X_train[selected_features])
X_test_step = sm.add_constant(X_test[selected_features], has_constant="add")[["const"] + selected_features]
ols_step_compare = sm.OLS(y_train, X_train_step).fit()
pred_step = ols_step_compare.predict(X_test_step)

#### 📌 Apa yang dilakukan di sini? — Evaluasi & Perbandingan Model

Ini bagian **evaluasi akhir**. Cell ini menghitung dua metrik utama untuk keempat model, dihitung dari data test (bukan data train, supaya adil):

- **RMSE (Root Mean Squared Error)** → rata-rata besar kesalahan prediksi (dalam satuan yang sama dengan Y, yaitu juta Rupiah). Semakin **kecil**, semakin baik.
- **R² (R-squared) pada data test** → seberapa besar variasi omzet di data test yang berhasil dijelaskan oleh masing-masing model. Semakin **besar** (mendekati 1), semakin baik.

Dengan tabel ini, kita bisa melihat model mana yang paling akurat memprediksi data baru — apakah model kompleks (OLS Full), model hasil seleksi (Stepwise), atau model regularisasi (Ridge/Lasso) yang tampil paling baik.

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "OLS Full (semua variabel)",
        "OLS Stepwise (variabel terpilih)",
        f"Ridge (alpha={ridge.alpha_:.4f})",
        f"Lasso (alpha={lasso.alpha_:.4f})",
    ],
    "RMSE_test": [
        np.sqrt(mean_squared_error(y_test, pred_ols)),
        np.sqrt(mean_squared_error(y_test, pred_step)),
        np.sqrt(mean_squared_error(y_test, pred_ridge)),
        np.sqrt(mean_squared_error(y_test, pred_lasso)),
    ],
    "R2_test": [
        r2_score(y_test, pred_ols),
        r2_score(y_test, pred_step),
        r2_score(y_test, pred_ridge),
        r2_score(y_test, pred_lasso),
    ],
})

print("=== PERBANDINGAN PERFORMA DENGAN PENALTI OPTIMAL (CROSS-VALIDATED) ===")
print(comparison.to_string(index=False))

#### 📌 Apa yang dilakukan di sini?

Cell terakhir ini secara spesifik melihat hasil **Lasso Regression**: variabel mana saja yang koefisiennya "dinolkan" oleh Lasso (dianggap tidak cukup penting untuk memprediksi omzet).

Ini menarik untuk dibandingkan dengan hasil **Metode Stepwise** sebelumnya — apakah kedua metode seleksi variabel yang berbeda (stepwise vs Lasso) menghasilkan kesimpulan variabel penting yang mirip atau berbeda.

In [ ]:
lasso_coef = pd.Series(lasso.coef_, index=numeric_candidate_features)
print("\nVariabel yang di-'nol'-kan oleh Lasso (dianggap tidak relevan):")
zeroed = lasso_coef[lasso_coef.abs() < 1e-6]
print(
    list(zeroed.index)
    if len(zeroed) > 0
    else "(tidak ada, semua variabel tetap punya koefisien non-nol)"
)

In [ ]:
lasso_coef = pd.Series(lasso.coef_, index=numeric_candidate_features).sort_values()

plt.figure(figsize=(10, 6))
lasso_coef.plot(kind="barh", color=(lasso_coef != 0).map({True: "steelblue", False: "salmon"}))
plt.axvline(x=0, color="black", linestyle="--", linewidth=0.8)
plt.title(f"Koefisien Fitur Lasso pada Alpha Optimal (alpha = {lasso.alpha_:.4f})")
plt.xlabel("Besaran Koefisien")
plt.ylabel("Variabel Prediktor")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()